<a href="https://colab.research.google.com/github/RonazAbbasi/ml-zoomcamp2026/blob/main/Homework4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score"]

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_full_train = df_full_train["converted"].values

X_train_df = df_train.drop(columns=["converted"])
X_val_df = df_val.drop(columns=["converted"])
X_full_df = df_full_train.drop(columns=["converted"])

print("Rows in dataset:", len(df))
print("\nQ1: ROC AUC feature importance")

auc_scores = {}
for col in numerical:
    auc = roc_auc_score(y_train, X_train_df[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -X_train_df[col])
    auc_scores[col] = auc
    print(col, round(auc, 3))

print("Best feature:", max(auc_scores, key=auc_scores.get))

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(X_train_df.to_dict(orient="records"))
X_val = dv.transform(X_val_df.to_dict(orient="records"))

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]

print("\nQ2: Model AUC")
print(round(roc_auc_score(y_val, y_pred), 3))

thresholds = np.arange(0.0, 1.01, 0.01)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    pred_binary = (y_pred >= t).astype(int)
    precisions.append(precision_score(y_val, pred_binary, zero_division=0))
    recalls.append(recall_score(y_val, pred_binary, zero_division=0))
    f1s.append(f1_score(y_val, pred_binary, zero_division=0))

valid = [i for i in range(len(thresholds)) if precisions[i] > 0 and recalls[i] > 0]
idx_intersection = min(valid, key=lambda i: abs(precisions[i] - recalls[i]))
idx_f1 = int(np.argmax(f1s))

print("\nQ3: Precision and Recall")
print("Threshold:", round(thresholds[idx_intersection], 2))
print("Precision:", round(precisions[idx_intersection], 3))
print("Recall:", round(recalls[idx_intersection], 3))

print("\nQ4: F1 score")
print("Best threshold:", round(thresholds[idx_f1], 2))
print("Best F1:", round(f1s[idx_f1], 3))

print("\nQ5: 5-Fold CV standard deviation")

kf = KFold(n_splits=5, shuffle=True, random_state=1)
cv_scores = []

for train_idx, val_idx in kf.split(X_full_df):
    train_fold = X_full_df.iloc[train_idx]
    val_fold = X_full_df.iloc[val_idx]
    y_train_fold = y_full_train[train_idx]
    y_val_fold = y_full_train[val_idx]

    dv_fold = DictVectorizer(sparse=True)
    X_train_fold = dv_fold.fit_transform(train_fold.to_dict(orient="records"))
    X_val_fold = dv_fold.transform(val_fold.to_dict(orient="records"))

    model_fold = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
    model_fold.fit(X_train_fold, y_train_fold)
    pred_fold = model_fold.predict_proba(X_val_fold)[:, 1]
    cv_scores.append(roc_auc_score(y_val_fold, pred_fold))

print("Fold scores:", [round(x, 3) for x in cv_scores])
print("Mean:", round(np.mean(cv_scores), 3))
print("Standard deviation:", round(np.std(cv_scores), 3))

print("\nQ6: Best C")

for C in [0.000001, 0.001, 1]:
    scores = []
    for train_idx, val_idx in kf.split(X_full_df):
        train_fold = X_full_df.iloc[train_idx]
        val_fold = X_full_df.iloc[val_idx]
        y_train_fold = y_full_train[train_idx]
        y_val_fold = y_full_train[val_idx]

        dv_fold = DictVectorizer(sparse=True)
        X_train_fold = dv_fold.fit_transform(train_fold.to_dict(orient="records"))
        X_val_fold = dv_fold.transform(val_fold.to_dict(orient="records"))

        model_fold = LogisticRegression(solver="liblinear", C=C, max_iter=1000)
        model_fold.fit(X_train_fold, y_train_fold)
        pred_fold = model_fold.predict_proba(X_val_fold)[:, 1]
        scores.append(roc_auc_score(y_val_fold, pred_fold))

    print("C:", C, "Mean AUC:", round(np.mean(scores), 3),
          "Std:", round(np.std(scores), 3))

Rows in dataset: 5000

Q1: ROC AUC feature importance
annual_income 0.602
number_of_courses_viewed 0.72
interaction_count 0.767
lead_score 0.788
Best feature: lead_score

Q2: Model AUC
0.729

Q3: Precision and Recall
Threshold: 0.59
Precision: 0.733
Recall: 0.759

Q4: F1 score
Best threshold: 0.55
Best F1: 0.767

Q5: 5-Fold CV standard deviation
Fold scores: [np.float64(0.742), np.float64(0.738), np.float64(0.714), np.float64(0.741), np.float64(0.721)]
Mean: 0.731
Standard deviation: 0.012

Q6: Best C
C: 1e-06 Mean AUC: 0.616 Std: 0.01
C: 0.001 Mean AUC: 0.744 Std: 0.013
C: 1 Mean AUC: 0.731 Std: 0.012
